In [3]:
# ============================================================
# FedPIGD-B v2 (FULL WORKING VERSION)
# ML Base Model: Federated Linear SVM (hinge-loss) head
# Backbone: MobileNetV2 (frozen) used ONLY as feature extractor
# Your novelty preserved: GA pipelines + signatures + anchoring + FL + FedAvg
# ============================================================

import os
import time
import copy
import random
import gc
import numpy as np
import tensorflow as tf

from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    matthews_corrcoef
)
from sklearn.preprocessing import label_binarize

# ============================================================
# 0) CONFIG
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TRAIN_DIR = "/Users/tanvir/Downloads/Research/Fish Recognition/Photos/Fish/train_data"
TEST_DIR  = "/Users/tanvir/Downloads/Research/Fish Recognition/Photos/Fish/test_data"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# LR for Linear SVM head (fast)
LR = 5e-3

NUM_CLIENTS = 4
FL_ROUNDS = 10
LOCAL_EPOCHS_PER_ROUND = 1

EV_POP = 10
EV_GENS = 6
EV_SUBSET = 64
MAX_PIPE_LEN = 5

ANCHOR_LAMBDA = 1e-3  # same anchoring idea

LOG_EVERY_BATCH = 20

# ============================================================
# LOGGING
# ============================================================
def ts(): return time.strftime("%H:%M:%S")
def log(msg): print(f"[{ts()}] [INFO] {msg}")
def phase(msg):
    print("\n" + "=" * 80)
    print(msg)
    print("=" * 80)

def cleanup():
    gc.collect()

# ============================================================
# 1) DATA LOADING
# ============================================================
def list_classes(train_dir):
    classes = sorted(d for d in os.listdir(train_dir)
                     if os.path.isdir(os.path.join(train_dir, d)))
    return classes, {c: i for i, c in enumerate(classes)}

def load_paths_labels(data_dir, class_to_idx):
    paths, labels = [], []
    for c, i in class_to_idx.items():
        cls_dir = os.path.join(data_dir, c)
        for f in os.listdir(cls_dir):
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                paths.append(os.path.join(cls_dir, f))
                labels.append(i)
    return np.array(paths), np.array(labels)

def split_clients(paths, labels, n):
    idx = np.random.permutation(len(paths))
    paths, labels = paths[idx], labels[idx]
    proportions = np.random.dirichlet([1.0] * n)
    sizes = (proportions * len(paths)).astype(int)
    sizes[-1] = len(paths) - sum(sizes[:-1])

    out, s = [], 0
    for i, sz in enumerate(sizes):
        out.append((paths[s:s+sz], labels[s:s+sz]))
        log(f"Client C{i+1}: {sz} samples")
        s += sz
    return out

phase("Loading dataset")
class_names, class_to_idx = list_classes(TRAIN_DIR)
K = len(class_names)
log(f"Detected {K} classes")

train_paths, train_labels = load_paths_labels(TRAIN_DIR, class_to_idx)
test_paths,  test_labels  = load_paths_labels(TEST_DIR,  class_to_idx)

clients = split_clients(train_paths, train_labels, NUM_CLIENTS)

# ============================================================
# 2) PIL PIPELINE OPS (non-differentiable)  <-- novelty preserved
# ============================================================
OP_SPACE = [
    "IDENTITY", "GRAYSCALE", "AUTO_CONTRAST", "EQUALIZE",
    "SHARPEN", "BRIGHTNESS", "CONTRAST", "BLUR"
]

def sample_op():
    op = random.choice(OP_SPACE)
    p = {}
    if op in ["SHARPEN", "BRIGHTNESS", "CONTRAST"]:
        p["factor"] = float(np.random.uniform(0.8, 1.5))
    if op == "BLUR":
        p["radius"] = float(np.random.uniform(0.2, 1.5))
    return (op, p)

def apply_op(img, op):
    name, p = op
    if name == "IDENTITY":
        return img
    if name == "GRAYSCALE":
        return ImageOps.grayscale(img).convert("RGB")
    if name == "AUTO_CONTRAST":
        return ImageOps.autocontrast(img)
    if name == "EQUALIZE":
        return ImageOps.equalize(img)
    if name == "SHARPEN":
        return ImageEnhance.Sharpness(img).enhance(p["factor"])
    if name == "BRIGHTNESS":
        return ImageEnhance.Brightness(img).enhance(p["factor"])
    if name == "CONTRAST":
        return ImageEnhance.Contrast(img).enhance(p["factor"])
    if name == "BLUR":
        return img.filter(ImageFilter.GaussianBlur(p["radius"]))
    return img

def apply_pipeline_np(x, pipe):
    img = Image.fromarray((np.clip(x, 0.0, 1.0) * 255).astype(np.uint8))
    for op in pipe:
        img = apply_op(img, op)
    return np.array(img).astype(np.float32) / 255.0

def tf_apply_pipeline(x, pipe):
    out = tf.numpy_function(lambda z: apply_pipeline_np(z, pipe), [x], tf.float32)
    out.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return out

# ============================================================
# 3) SAFE DECODE
# ============================================================
def decode_and_resize(p):
    img = tf.image.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMAGE_SIZE)
    return tf.cast(img, tf.float32) / 255.0

# ============================================================
# 4) FROZEN FEATURE EXTRACTOR (MobileNetV2)  <-- stable for your dataset
# ============================================================
phase("Building frozen feature extractor (MobileNetV2)")

def build_feature_extractor():
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )
    base.trainable = False
    inp = tf.keras.Input(shape=(224, 224, 3))
    x = base(inp, training=False)
    feat = tf.keras.layers.GlobalAveragePooling2D()(x)  # (B, D)
    return tf.keras.Model(inp, feat, name="mobilenetv2_feat")

feat_model = build_feature_extractor()

# infer feature dimension D
_dummy = tf.zeros([1, 224, 224, 3], dtype=tf.float32)
D = int(feat_model(_dummy).shape[-1])
log(f"Feature dim D = {D}")

# ============================================================
# 5) GA EVOLUTION (SEPARABILITY PROXY)  <-- novelty preserved
# ============================================================
phase("Evolving pipelines (GA + separability proxy)")

def separability_proxy(f, l):
    sb, sw = 0.0, 0.0
    mu = f.mean(axis=0, keepdims=True)
    for c in np.unique(l):
        Xc = f[l == c]
        if len(Xc) < 2:
            continue
        mc = Xc.mean(axis=0, keepdims=True)
        sb += len(Xc) * np.sum((mc - mu) ** 2)
        sw += np.sum((Xc - mc) ** 2)
    return float(sb / (sw + 1e-8))

def evolve_pipeline(paths, labels):
    pop = [[sample_op() for _ in range(random.randint(1, MAX_PIPE_LEN))]
           for _ in range(EV_POP)]
    best_pipe, best_score = None, -1e9

    for g in range(EV_GENS):
        for pipe in pop:
            n = min(EV_SUBSET, len(paths))
            idx = np.random.choice(len(paths), n, replace=False) if len(paths) > n else np.arange(len(paths))

            xs = []
            for p in paths[idx]:
                x = decode_and_resize(tf.constant(p))
                x = tf_apply_pipeline(x, pipe)
                xs.append(x.numpy())
            xs = np.array(xs, dtype=np.float32)

            feats = feat_model.predict(xs, verbose=0)
            score = separability_proxy(feats, labels[idx]) - 0.02 * len(pipe)

            if score > best_score:
                best_score = score
                best_pipe = pipe

            del xs, feats
            cleanup()

        log(f"GA Gen {g+1}/{EV_GENS} | BestScore={best_score:.4f}")

    return best_pipe

client_pipelines = [evolve_pipeline(cp, cl) for cp, cl in clients]

# ============================================================
# 6) PIPELINE SIGNATURES  <-- novelty preserved
# ============================================================
phase("Building pipeline signatures")

def pipeline_signature(pipe):
    hist = np.zeros(len(OP_SPACE), dtype=np.float32)
    factors, radii = [], []
    for (op, p) in pipe:
        hist[OP_SPACE.index(op)] += 1.0
        if "factor" in p: factors.append(float(p["factor"]))
        if "radius" in p: radii.append(float(p["radius"]))
    f_mean = np.mean(factors) if factors else 0.0
    f_std  = np.std(factors) if factors else 0.0
    r_mean = np.mean(radii) if radii else 0.0
    r_std  = np.std(radii) if radii else 0.0
    length = float(len(pipe))
    runtime_proxy = length
    return np.concatenate([hist, np.array([f_mean, f_std, r_mean, r_std, length, runtime_proxy], np.float32)])

client_sigs = np.stack([pipeline_signature(p) for p in client_pipelines], axis=0)
log(f"Signature dim = {client_sigs.shape[1]}")
log(f"Signatures shape = {client_sigs.shape}")

# ============================================================
# 7) FEDERATED LEARNING + SIGNATURE-CONDITIONED ANCHORING
# ML Model = Linear SVM head trained with hinge loss
# ============================================================
phase("Federated training (Linear SVM head) + anchoring")

def one_hot(y, K):
    return tf.one_hot(tf.cast(y, tf.int32), depth=K)

def hinge_loss_multiclass(logits, y_onehot):
    # y_onehot in {0,1}
    # Convert to {-1,+1} for hinge: y = 2*onehot - 1
    y = 2.0 * y_onehot - 1.0
    # hinge = mean(max(0, 1 - y*logit)) across classes
    return tf.reduce_mean(tf.nn.relu(1.0 - y * logits))

def flatten_wb(W, b):
    return tf.concat([tf.reshape(W, [-1]), tf.reshape(b, [-1])], axis=0)

def unflatten_wb(vec, D, K):
    w_size = D * K
    W = tf.reshape(vec[:w_size], [D, K])
    b = tf.reshape(vec[w_size:w_size + K], [K])
    return W, b

def fedavg_vectors(vecs, sizes):
    total = float(sum(sizes))
    acc = np.zeros_like(vecs[0], dtype=np.float32)
    for v, sz in zip(vecs, sizes):
        acc += (sz / total) * v
    return acc

class ConditionerRidge:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.models = None
    def fit(self, S, A):
        self.models = [Ridge(alpha=self.alpha).fit(S, A[:, j]) for j in range(A.shape[1])]
    def predict(self, s):
        s = s.reshape(1, -1)
        return np.array([m.predict(s)[0] for m in self.models], dtype=np.float32)

conditioner = ConditionerRidge(alpha=1.0)

# Global SVM parameters (W,b) initialized small
global_W = tf.Variable(tf.random.normal([D, K], stddev=0.01, seed=SEED), trainable=True, dtype=tf.float32)
global_b = tf.Variable(tf.zeros([K], dtype=tf.float32), trainable=True)

global_vec = flatten_wb(global_W, global_b).numpy().astype(np.float32)

# Anchors: signature -> predicted SVM parameter vector
anchors = [global_vec.copy() for _ in range(NUM_CLIENTS)]
log("Anchors initialized from global Linear SVM vector.")

def make_client_dataset(paths, labels, pipeline, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(2000, len(paths)), seed=SEED, reshuffle_each_iteration=True)

    def map_fn(x, y, pipe=pipeline):
        img = decode_and_resize(x)
        img = tf_apply_pipeline(img, pipe)   # client-specific GA pipeline
        return img, y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def svm_train_step(batch_imgs, batch_y, W, b, opt, anchor_vec, lam):
    feats = feat_model(batch_imgs, training=False)
    logits = tf.matmul(feats, W) + b

    y_oh = tf.one_hot(tf.cast(batch_y, tf.int32), depth=K)
    y = 2.0 * y_oh - 1.0
    loss_h = tf.reduce_mean(tf.nn.relu(1.0 - y * logits))

    cur_vec = tf.concat([tf.reshape(W, [-1]), tf.reshape(b, [-1])], axis=0)
    anchor_tf = tf.convert_to_tensor(anchor_vec, dtype=tf.float32)
    anchor_loss = tf.reduce_sum(tf.square(cur_vec - anchor_tf))

    loss = loss_h + lam * anchor_loss

    grads = tf.gradients(loss, [W, b])
    opt.apply_gradients(zip(grads, [W, b]))

    return loss

def svm_train_step(batch_imgs, batch_y, W, b, opt, anchor_vec, lam):
    with tf.GradientTape() as tape:
        feats = feat_model(batch_imgs, training=False)  # frozen backbone
        logits = tf.matmul(feats, W) + b

        # one-vs-rest hinge loss
        y_oh = tf.one_hot(tf.cast(batch_y, tf.int32), depth=K)
        y = 2.0 * y_oh - 1.0
        loss_h = tf.reduce_mean(tf.nn.relu(1.0 - y * logits))

        # anchoring (unchanged)
        cur_vec = tf.concat(
            [tf.reshape(W, [-1]), tf.reshape(b, [-1])], axis=0
        )
        anchor_tf = tf.convert_to_tensor(anchor_vec, dtype=tf.float32)
        anchor_loss = tf.reduce_sum(tf.square(cur_vec - anchor_tf))

        loss = loss_h + lam * anchor_loss

    grads = tape.gradient(loss, [W, b])
    opt.apply_gradients(zip(grads, [W, b]))

    return loss

for r in range(1, FL_ROUNDS + 1):
    phase(f"FL ROUND {r}/{FL_ROUNDS}")

    local_vecs = []
    sizes = []
    local_centers = []

    for ci, (cp, cl) in enumerate(clients, start=1):
        log(f"Client C{ci}: sync global SVM params")
        W = tf.Variable(global_W.read_value(), trainable=True)
        b = tf.Variable(global_b.read_value(), trainable=True)

        ds = make_client_dataset(cp, cl, client_pipelines[ci-1], shuffle=True)

        opt = tf.keras.optimizers.SGD(learning_rate=LR, momentum=0.9)

        steps = 0
        total_loss = 0.0

        for epoch in range(LOCAL_EPOCHS_PER_ROUND):
            for xb, yb in ds:
                loss = svm_train_step(xb, yb, W, b, opt, anchors[ci-1], ANCHOR_LAMBDA)
                total_loss += float(loss.numpy())
                steps += 1
                if steps % LOG_EVERY_BATCH == 0:
                    log(f"[Round {r}] C{ci} batch={steps} | loss={total_loss/steps:.4f}")

        avg_loss = total_loss / max(steps, 1)
        log(f"Client C{ci}: epoch done | avg loss={avg_loss:.4f}")

        vec = flatten_wb(W, b).numpy().astype(np.float32)
        local_vecs.append(vec)
        sizes.append(len(cp))
        local_centers.append(vec)

        del ds, opt, W, b
        cleanup()

    # FedAvg on SVM parameters
    log("Server: FedAvg aggregate Linear SVM params")
    new_global_vec = fedavg_vectors(local_vecs, sizes)

    # assign back to global variables
    newW, newb = unflatten_wb(tf.convert_to_tensor(new_global_vec, tf.float32), D, K)
    global_W.assign(newW)
    global_b.assign(newb)

    log("Server: global Linear SVM updated.")

    # Update conditioner: signature -> anchor vector
    log("Server: update conditioner (signature -> SVM anchor)")
    A = np.stack(local_centers, axis=0)
    conditioner.fit(client_sigs, A)
    anchors = [conditioner.predict(client_sigs[i]) for i in range(NUM_CLIENTS)]
    log("Server: anchors refreshed for next round.")

    del local_vecs, local_centers, A
    cleanup()

# ============================================================
# FINAL EVALUATION (ALL METRICS + FULL LOGS)
# (SVM gives logits; we use softmax(logits) as pseudo-prob for AUC/logloss)
# ============================================================
phase("FINAL EVALUATION: Test set (no client preprocessing)")

def make_test_dataset(paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def map_fn(x, y):
        img = decode_and_resize(x)
        return img, y
    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

test_ds = make_test_dataset(test_paths, test_labels)

all_logits = []
all_true = []

for xb, yb in test_ds:
    feats = feat_model(xb, training=False).numpy()
    logits = feats @ global_W.numpy() + global_b.numpy()
    all_logits.append(logits)
    all_true.append(yb.numpy())

all_logits = np.concatenate(all_logits, axis=0)
test_labels_np = np.concatenate(all_true, axis=0)

# pseudo probabilities (for logloss/AUC)
pred_probs = tf.nn.softmax(all_logits, axis=1).numpy()
preds = np.argmax(pred_probs, axis=1)

acc  = accuracy_score(test_labels_np, preds)
prec = precision_score(test_labels_np, preds, average="weighted", zero_division=0)
rec  = recall_score(test_labels_np, preds, average="weighted", zero_division=0)
f1   = f1_score(test_labels_np, preds, average="weighted", zero_division=0)
ll   = log_loss(test_labels_np, pred_probs)

y_true_bin = label_binarize(test_labels_np, classes=list(range(K)))

roc_auc = roc_auc_score(
    y_true_bin,
    pred_probs,
    average="macro",
    multi_class="ovr"
)

pr_auc = average_precision_score(
    y_true_bin,
    pred_probs,
    average="macro"
)

mcc = matthews_corrcoef(test_labels_np, preds)

log(f"Accuracy     = {acc:.4f}")
log(f"Precision    = {prec:.4f} (weighted)")
log(f"Recall       = {rec:.4f} (weighted)")
log(f"F1-score     = {f1:.4f} (weighted)")
log(f"ROC-AUC      = {roc_auc:.4f} (macro, OvR)")
log(f"PR-AUC       = {pr_auc:.4f} (macro)")
log(f"Log-Loss     = {ll:.4f}")
log(f"MCC Score    = {mcc:.4f}")

print("\nClassification Report:")
print(classification_report(
    test_labels_np,
    preds,
    target_names=class_names,
    digits=4,
    zero_division=0
))

cleanup()



Loading dataset
[23:30:58] [INFO] Detected 23 classes
[23:30:58] [INFO] Client C1: 2648 samples
[23:30:58] [INFO] Client C2: 158 samples
[23:30:58] [INFO] Client C3: 1364 samples
[23:30:58] [INFO] Client C4: 765 samples

Building frozen feature extractor (MobileNetV2)
[23:30:58] [INFO] Feature dim D = 1280

Evolving pipelines (GA + separability proxy)
[23:31:15] [INFO] GA Gen 1/6 | BestScore=1.1654
[23:31:30] [INFO] GA Gen 2/6 | BestScore=1.1654
[23:31:47] [INFO] GA Gen 3/6 | BestScore=1.1654
[23:32:02] [INFO] GA Gen 4/6 | BestScore=1.1654
[23:32:17] [INFO] GA Gen 5/6 | BestScore=1.1654
[23:32:32] [INFO] GA Gen 6/6 | BestScore=1.1654
[23:32:50] [INFO] GA Gen 1/6 | BestScore=0.9074
[23:33:07] [INFO] GA Gen 2/6 | BestScore=0.9803
[23:33:25] [INFO] GA Gen 3/6 | BestScore=0.9803
[23:33:42] [INFO] GA Gen 4/6 | BestScore=0.9803
[23:33:59] [INFO] GA Gen 5/6 | BestScore=0.9803
[23:34:16] [INFO] GA Gen 6/6 | BestScore=0.9803
[23:34:32] [INFO] GA Gen 1/6 | BestScore=0.8458
[23:34:47] [INFO] GA 

2026-01-05 23:38:47.640245: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[23:38:47] [INFO] Client C1: epoch done | avg loss=0.1704
[23:38:47] [INFO] Client C2: sync global SVM params
[23:38:53] [INFO] Client C2: epoch done | avg loss=0.6353
[23:38:53] [INFO] Client C3: sync global SVM params
[23:39:15] [INFO] [Round 1] C3 batch=20 | loss=0.2776
[23:39:36] [INFO] [Round 1] C3 batch=40 | loss=0.2268


2026-01-05 23:39:39.066989: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[23:39:39] [INFO] Client C3: epoch done | avg loss=0.2215
[23:39:39] [INFO] Client C4: sync global SVM params
[23:39:59] [INFO] [Round 1] C4 batch=20 | loss=0.2766
[23:40:03] [INFO] Client C4: epoch done | avg loss=0.2616
[23:40:03] [INFO] Server: FedAvg aggregate Linear SVM params
[23:40:03] [INFO] Server: global Linear SVM updated.
[23:40:03] [INFO] Server: update conditioner (signature -> SVM anchor)
[23:40:11] [INFO] Server: anchors refreshed for next round.

FL ROUND 2/10
[23:40:11] [INFO] Client C1: sync global SVM params
[23:40:34] [INFO] [Round 2] C1 batch=20 | loss=0.1035
[23:40:54] [INFO] [Round 2] C1 batch=40 | loss=0.0943
[23:41:15] [INFO] [Round 2] C1 batch=60 | loss=0.0879
[23:41:36] [INFO] [Round 2] C1 batch=80 | loss=0.0834
[23:41:38] [INFO] Client C1: epoch done | avg loss=0.0826
[23:41:39] [INFO] Client C2: sync global SVM params
[23:41:44] [INFO] Client C2: epoch done | avg loss=0.1211
[23:41:44] [INFO] Client C3: sync global SVM params
[23:42:04] [INFO] [Round 2] C3

...

[23:42:27] [INFO] Client C3: epoch done | avg loss=0.0947
[23:42:27] [INFO] Client C4: sync global SVM params
[23:42:48] [INFO] [Round 2] C4 batch=20 | loss=0.1065
[23:42:52] [INFO] Client C4: epoch done | avg loss=0.1052
[23:42:52] [INFO] Server: FedAvg aggregate Linear SVM params
[23:42:52] [INFO] Server: global Linear SVM updated.
[23:42:52] [INFO] Server: update conditioner (signature -> SVM anchor)
[23:43:00] [INFO] Server: anchors refreshed for next round.

FL ROUND 3/10
[23:43:00] [INFO] Client C1: sync global SVM params
[23:43:20] [INFO] [Round 3] C1 batch=20 | loss=0.0684
[23:43:40] [INFO] [Round 3] C1 batch=40 | loss=0.0660
[23:43:59] [INFO] [Round 3] C1 batch=60 | loss=0.0640
[23:44:20] [INFO] [Round 3] C1 batch=80 | loss=0.0622
[23:44:22] [INFO] Client C1: epoch done | avg loss=0.0617
[23:44:22] [INFO] Client C2: sync global SVM params
[23:44:28] [INFO] Client C2: epoch done | avg loss=0.0822
[23:44:28] [INFO] Client C3: sync global SVM params
[23:44:50] [INFO] [Round 3] C3

2026-01-05 23:48:04.954009: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[23:48:04] [INFO] Client C3: epoch done | avg loss=0.0553
[23:48:05] [INFO] Client C4: sync global SVM params
[23:48:25] [INFO] [Round 4] C4 batch=20 | loss=0.0618
[23:48:29] [INFO] Client C4: epoch done | avg loss=0.0619
[23:48:29] [INFO] Server: FedAvg aggregate Linear SVM params
[23:48:29] [INFO] Server: global Linear SVM updated.
[23:48:29] [INFO] Server: update conditioner (signature -> SVM anchor)
[23:48:36] [INFO] Server: anchors refreshed for next round.

FL ROUND 5/10
[23:48:36] [INFO] Client C1: sync global SVM params
[23:48:57] [INFO] [Round 5] C1 batch=20 | loss=0.0454
[23:49:16] [INFO] [Round 5] C1 batch=40 | loss=0.0450
[23:49:35] [INFO] [Round 5] C1 batch=60 | loss=0.0447
[23:49:55] [INFO] [Round 5] C1 batch=80 | loss=0.0443
[23:49:57] [INFO] Client C1: epoch done | avg loss=0.0439
[23:49:57] [INFO] Client C2: sync global SVM params
[23:50:03] [INFO] Client C2: epoch done | avg loss=0.0620
[23:50:03] [INFO] Client C3: sync global SVM params
[23:50:23] [INFO] [Round 5] C3

2026-01-05 23:58:42.771819: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[23:58:42] [INFO] Client C3: epoch done | avg loss=0.0328
[23:58:42] [INFO] Client C4: sync global SVM params
[23:59:02] [INFO] [Round 8] C4 batch=20 | loss=0.0401
[23:59:06] [INFO] Client C4: epoch done | avg loss=0.0404
[23:59:06] [INFO] Server: FedAvg aggregate Linear SVM params
[23:59:06] [INFO] Server: global Linear SVM updated.
[23:59:06] [INFO] Server: update conditioner (signature -> SVM anchor)
[23:59:13] [INFO] Server: anchors refreshed for next round.

FL ROUND 9/10
[23:59:13] [INFO] Client C1: sync global SVM params
[23:59:33] [INFO] [Round 9] C1 batch=20 | loss=0.0270
[23:59:52] [INFO] [Round 9] C1 batch=40 | loss=0.0270
[00:00:13] [INFO] [Round 9] C1 batch=60 | loss=0.0272
[00:00:34] [INFO] [Round 9] C1 batch=80 | loss=0.0274
[00:00:36] [INFO] Client C1: epoch done | avg loss=0.0270
[00:00:36] [INFO] Client C2: sync global SVM params
[00:00:42] [INFO] Client C2: epoch done | avg loss=0.0462
[00:00:42] [INFO] Client C3: sync global SVM params
[00:01:02] [INFO] [Round 9] C3